In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import pickle
import glob
import os

In [ ]:
csv_files = glob.glob('capture_*.csv')
print(f"Found {len(csv_files)} CSV files: {csv_files}")

Found 74 CSV files: ['capture_034.csv', 'capture_060.csv', 'capture_033.csv', 'capture_057.csv', 'capture_001.csv', 'capture_071.csv', 'capture_022.csv', 'capture_073.csv', 'capture_047.csv', 'capture_015.csv', 'capture_072.csv', 'capture_027.csv', 'capture_024.csv', 'capture_011.csv', 'capture_067.csv', 'capture_058.csv', 'capture_052.csv', 'capture_020.csv', 'capture_069.csv', 'capture_026.csv', 'capture_031.csv', 'capture_066.csv', 'capture_025.csv', 'capture_064.csv', 'capture_013.csv', 'capture_070.csv', 'capture_043.csv', 'capture_019.csv', 'capture_055.csv', 'capture_007.csv', 'capture_041.csv', 'capture_054.csv', 'capture_014.csv', 'capture_059.csv', 'capture_021.csv', 'capture_063.csv', 'capture_037.csv', 'capture_062.csv', 'capture_023.csv', 'capture_028.csv', 'capture_010.csv', 'capture_032.csv', 'capture_006.csv', 'capture_061.csv', 'capture_050.csv', 'capture_035.csv', 'capture_009.csv', 'capture_040.csv', 'capture_051.csv', 'capture_016.csv', 'capture_003.csv', 'capture_0

In [ ]:
# Load and combine CSV files
dfs = [pd.read_csv(file) for file in csv_files]
combined_df = pd.concat(dfs, ignore_index=True)

In [ ]:
combined_df = combined_df.dropna()
print(f"Dataset shape after dropping all NaN rows: {combined_df.shape}")

Dataset shape after dropping all NaN rows: (4348155, 13)


In [ ]:
duplicate_count = combined_df.duplicated().sum()
print(f"Duplicate rows in combined_df: {duplicate_count}")
if duplicate_count > 0:
    combined_df = combined_df.drop_duplicates()
    print(f"Dataset shape after dropping duplicates: {combined_df.shape}")

Duplicate rows in combined_df: 1856383
Dataset shape after dropping duplicates: (2491772, 13)


In [ ]:
combined_df.head()

,timestamp,src_mac,dst_mac,src_ip,dst_ip,protocol,ttl,total_length,src_port,dst_port,tcp_flags,window,payload_size
19,1.745268e+09,b8:1e:a4:d4:bb:21,42:88:2f:4d:0a:62,134.88.138.163,23.38.112.50,6,128,108,58213.0,443.0,PA,511.0,68
20,1.745268e+09,b8:1e:a4:d4:bb:21,42:88:2f:4d:0a:62,134.88.138.163,23.201.34.138,6,128,108,53800.0,443.0,PA,4092.0,68
28,1.745268e+09,54:d7:e3:d2:11:90,b8:1e:a4:d4:bb:21,23.38.112.50,134.88.138.163,6,51,266,443.0,58213.0,PA,501.0,226
29,1.745268e+09,54:d7:e3:d2:11:90,b8:1e:a4:d4:bb:21,23.38.112.50,134.88.138.163,6,51,278,443.0,58213.0,PA,501.0,238
30,1.745268e+09,b8:1e:a4:d4:bb:21,42:88:2f:4d:0a:62,134.88.138.163,23.38.112.50,6,128,40,58213.0,443.0,A,510.0,0


In [ ]:
combined_df.to_csv('combined_network_traffic.csv', index=False)

In [ ]:
#Step 2: Preprocess the data
# Handle data types
combined_df['src_port'] = combined_df['src_port'].astype(int)
combined_df['dst_port'] = combined_df['dst_port'].astype(int)
combined_df['window'] = combined_df['window'].astype(int)
combined_df['payload_size'] = combined_df['payload_size'].astype(int)
combined_df['tcp_flags'] = combined_df['tcp_flags'].replace('', 'NONE')

# Derive time_diff for temporal patterns
combined_df['timestamp'] = pd.to_datetime(combined_df['timestamp'], unit='s', errors='coerce')
combined_df['time_diff'] = combined_df['timestamp'].diff().dt.total_seconds().fillna(0)

In [ ]:
# Verify required columns
required_columns = ['ttl', 'total_length', 'window', 'payload_size', 'time_diff', 'protocol', 'tcp_flags']
missing_columns = [col for col in required_columns if col not in combined_df.columns]
if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

In [ ]:
# Step 3: Feature selection and normalization
numerical_features = ['ttl', 'total_length', 'window', 'payload_size', 'time_diff']
categorical_features = ['protocol', 'tcp_flags']

# Normalize numerical features
scaler = StandardScaler()
df_numerical = scaler.fit_transform(combined_df[numerical_features])
df_numerical = pd.DataFrame(df_numerical, columns=numerical_features, index=combined_df.index)
print(f"df_numerical shape: {df_numerical.shape}")

# One-hot encode categorical features
df_categorical = pd.get_dummies(combined_df[categorical_features])
print(f"df_categorical shape: {df_categorical.shape}")

# Combine features
df_processed = pd.concat([df_numerical, df_categorical], axis=1)
print(f"Processed dataset shape: {df_processed.shape}")

# Verify row consistency
if len(df_processed) != len(combined_df):
    raise ValueError(f"Row mismatch: df_processed ({len(df_processed)}) vs combined_df ({len(combined_df)})")

print("Preprocessing completed")

df_numerical shape: (2491772, 5)
df_categorical shape: (2491772, 9)
Processed dataset shape: (2491772, 14)
Preprocessing completed


In [ ]:
# Save scaler
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

In [ ]:
# Step 4: Train Isolation Forest model
model = IsolationForest(n_estimators=100, contamination=0.01, random_state=42)
model.fit(df_processed)

# Save the trained model
with open('isolation_forest_model.pkl', 'wb') as f:
    pickle.dump(model, f)

print("Model training completed. Saved model as 'isolation_forest_model.pkl' and scaler as 'scaler.pkl'.")

Model training completed. Saved model as 'isolation_forest_model.pkl' and scaler as 'scaler.pkl'.


In [ ]:
import joblib
os.makedirs('models', exist_ok=True)

# Convert protocol to string for consistent one-hot encoding
combined_df['protocol'] = combined_df['protocol'].astype(str)

# One-hot encode categorical features
print("\nOne-hot encoding information:")
for feature in categorical_features:
    unique_values = combined_df[feature].unique()
    print(f"{feature} unique values: {unique_values}")

df_categorical = pd.get_dummies(combined_df[categorical_features])
dummy_columns = df_categorical.columns.tolist()
print(f"One-hot encoded columns ({len(dummy_columns)}): {dummy_columns}")

# Save dummy columns for later use
joblib.dump(dummy_columns, 'models/dummy_columns.pkl')
print("Saved one-hot encoded column names to 'models/dummy_columns.pkl'")


One-hot encoding information:
protocol unique values: ['6']
tcp_flags unique values: ['PA' 'A' 'S' 'SA' 'FA' 'RA' 'R' 'FPA']
One-hot encoded columns (9): ['protocol_6', 'tcp_flags_A', 'tcp_flags_FA', 'tcp_flags_FPA', 'tcp_flags_PA', 'tcp_flags_R', 'tcp_flags_RA', 'tcp_flags_S', 'tcp_flags_SA']
Saved one-hot encoded column names to 'models/dummy_columns.pkl'
